# HiddenExtension V3 결과 분석

Wind-aware IDW + Random Forest 기반 실험 결과 분석

분석 구성:
1. 전체 실험 결과 요약
2. Feature Importance 분석
3. Hidden vs LUR vs Temporal 기여도
4. IDW 방법 비교 (wind / idw / nearest)
5. 잔차 분석 (최고 모델 기준)
6. Grid PM10 예측 지도

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as stats

_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'config.py')):
    V3_DIR = _cwd
elif os.path.isfile(os.path.join(_cwd, 'HiddenExtension_V3', 'config.py')):
    V3_DIR = os.path.join(_cwd, 'HiddenExtension_V3')
else:
    V3_DIR = '/workspace/ST-GNN Modeling/HiddenExtension_V3'

sys.path.insert(0, V3_DIR)
CKPT_DIR = os.path.join(V3_DIR, 'checkpoints')
print(f'V3_DIR: {V3_DIR}')

## 1. 전체 실험 결과 요약

In [ ]:
# 모든 metrics.json 로드
files = glob.glob(os.path.join(CKPT_DIR, '**/metrics.json'), recursive=True)
records = [json.load(open(f)) for f in files]
df_all = pd.DataFrame(records).sort_values('test_mae').reset_index(drop=True)
df_all.index += 1

stgnn_mae = df_all['stgnn_mae'].iloc[0]
df_all['vs_baseline'] = df_all['test_mae'] - stgnn_mae
df_all['improved']    = df_all['vs_baseline'] < 0

show_cols = ['exp_id', 'use_hidden', 'idw_method', 'pca_k',
             'use_lur', 'use_population', 'use_temporal',
             'n_features', 'val_mae', 'test_mae', 'vs_baseline', 'test_r2', 'elapsed_min']

def highlight_row(row):
    if row['exp_id'] == df_all.iloc[0]['exp_id']:  # best
        return ['background-color: #d4edda'] * len(row)
    if row['improved']:
        return ['background-color: #eaf4fb'] * len(row)
    return [''] * len(row)

print(f'ST-GNN baseline MAE: {stgnn_mae:.4f}')
print(f'baseline 개선 실험:  {df_all["improved"].sum()}/{len(df_all)}개')
df_all[[c for c in show_cols if c in df_all.columns]].style \
    .apply(highlight_row, axis=1) \
    .format({'test_mae': '{:.4f}', 'val_mae': '{:.4f}',
             'vs_baseline': '{:+.4f}', 'test_r2': '{:.4f}'})

In [ ]:
# 실험 그룹별 바차트 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71' if v < 0 else '#e74c3c' for v in df_all['vs_baseline']]

ax = axes[0]
ax.barh(df_all['exp_id'][::-1], df_all['test_mae'][::-1],
        color=colors[::-1], edgecolor='white', height=0.7)
ax.axvline(stgnn_mae, color='black', linewidth=2, linestyle='--',
           label=f'ST-GNN baseline ({stgnn_mae:.4f})')
ax.set_xlabel('Test MAE', fontsize=11)
ax.set_title('Test MAE 비교', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
for i, (_, row) in enumerate(df_all.iloc[::-1].iterrows()):
    ax.text(row['test_mae'] + 0.01, i, f"{row['test_mae']:.4f}", va='center', fontsize=8)

ax = axes[1]
ax.barh(df_all['exp_id'][::-1], df_all['vs_baseline'][::-1],
        color=colors[::-1], edgecolor='white', height=0.7)
ax.axvline(0, color='black', linewidth=1.5)
ax.set_xlabel('vs ST-GNN baseline (음수=개선)', fontsize=11)
ax.set_title('베이스라인 대비 MAE 변화', fontsize=12)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('HiddenExtension V3 — 전체 실험 결과', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Feature Importance 분석

In [ ]:
# 최고 성능 모델의 feature importance
best_exp = df_all.iloc[0]['exp_id']
fi_path  = os.path.join(CKPT_DIR, best_exp, 'feature_importance.csv')

fi = pd.read_csv(fi_path)
top_n = min(30, len(fi))
fi_top = fi.head(top_n)

# 피처 그룹 색상 구분
def feat_color(name):
    if name.startswith('h_') or name.startswith('pca_'):
        return '#3498db'  # hidden
    if name in ['NDVI','IBI','buildings_%','greenspace_%','road_struc_%','river_zone_%',
                'elev_mean','sum_area','sum_height','population']:
        return '#e67e22'  # LUR
    return '#2ecc71'  # temporal

colors_fi = [feat_color(n) for n in fi_top['feature']]

fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.3)))
ax.barh(fi_top['feature'][::-1], fi_top['importance'][::-1],
        color=colors_fi[::-1], edgecolor='white', height=0.8)
ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title(f'Top-{top_n} Feature Importance — {best_exp}', fontsize=12)

# 범례
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#3498db', label='Hidden dimension'),
    Patch(color='#e67e22', label='LUR / spatial'),
    Patch(color='#2ecc71', label='Temporal'),
], fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# 그룹별 합산
fi['group'] = fi['feature'].apply(lambda x:
    'Hidden' if (x.startswith('h_') or x.startswith('pca_')) else
    ('Temporal' if x in ['hour_sin','hour_cos','month_sin','month_cos',
                          'doy_sin','doy_cos','is_weekend','is_winter','is_spring']
     else 'LUR/Spatial'))
group_sum = fi.groupby('group')['importance'].sum().sort_values(ascending=False)
print('\n피처 그룹별 중요도 합산:')
for g, v in group_sum.items():
    bar = '█' * int(v * 50)
    print(f'  {g:<15}: {v:.4f}  ({v:.1%})  {bar}')

In [ ]:
# Hidden dimension별 중요도 분포 (hidden 사용 실험 모두)
hidden_exps = df_all[df_all['use_hidden'] == True]['exp_id'].tolist()

if hidden_exps:
    # RF-HST 기준으로 hidden dim 중요도 히트맵
    main_exp = 'RF-HST' if 'RF-HST' in hidden_exps else hidden_exps[0]
    fi_main  = pd.read_csv(os.path.join(CKPT_DIR, main_exp, 'feature_importance.csv'))
    h_dims   = fi_main[fi_main['feature'].str.startswith('h_')].copy()
    h_dims['dim'] = h_dims['feature'].str.replace('h_','').astype(int)
    h_dims = h_dims.sort_values('dim')

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.bar(h_dims['dim'], h_dims['importance'], color='#3498db', width=0.8)
    ax.set_xlabel('Hidden dimension index', fontsize=11)
    ax.set_ylabel('Importance', fontsize=11)
    ax.set_title(f'Hidden Vector 차원별 중요도 — {main_exp}', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    top5_dims = h_dims.nlargest(5, 'importance')[['dim','importance']]
    print('Top-5 hidden dimensions:')
    print(top5_dims.to_string(index=False))

## 3. Hidden / LUR / Temporal 기여도 비교

In [ ]:
compare_pairs = [
    ('RF-S',   'RF-ST',   'Temporal 추가'),
    ('RF-S',   'RF-HST',  'Hidden 추가'),
    ('RF-ST',  'RF-HST',  'ST → Hidden+S+T'),
    ('RF-H',   'RF-HT',   'Temporal 추가 (H만)'),
    ('RF-HT',  'RF-HST',  'LUR 추가'),
    ('RF-HST', 'RF-HSTP', 'Population 추가'),
]

mae_map = dict(zip(df_all['exp_id'], df_all['test_mae']))
mae_map['baseline'] = stgnn_mae

print(f"{'비교':<30} {'A MAE':>8} {'B MAE':>8} {'개선폭':>8}")
print('-' * 60)
for a, b, label in compare_pairs:
    if a in mae_map and b in mae_map:
        delta = mae_map[b] - mae_map[a]
        sign  = '↑개선' if delta < 0 else '↓악화'
        print(f"{label:<30} {mae_map[a]:>8.4f} {mae_map[b]:>8.4f} "
              f"{sign} {abs(delta):.4f}")

## 4. IDW 방법 비교

In [ ]:
idw_exps = ['RF-HST', 'RF-HST-IDW', 'RF-HST-Nearest']
idw_data = df_all[df_all['exp_id'].isin(idw_exps)][['exp_id','idw_method','test_mae','test_r2']]

print('IDW 방법 비교:')
print(idw_data.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(idw_data))
bars = ax.bar(x, idw_data['test_mae'],
              color=['#e74c3c','#3498db','#95a5a6'], width=0.5)
ax.axhline(stgnn_mae, color='black', linestyle='--', linewidth=1.5,
           label=f'ST-GNN baseline ({stgnn_mae:.4f})')
ax.set_xticks(x)
ax.set_xticklabels([f"{r['exp_id']}\n({r['idw_method']})"
                    for _, r in idw_data.iterrows()], fontsize=9)
ax.set_ylabel('Test MAE', fontsize=11)
ax.set_title('Grid Hidden 생성 방법 비교', fontsize=12)
ax.legend(fontsize=9)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.4f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 최고 모델 잔차 분석

In [ ]:
import joblib
sys.path.insert(0, V3_DIR)
from feature_builder import FeatureBuilder, load_split_data, build_temporal_features, build_lur_features, flatten_station_data
from config import NDVI_PATH, IBI_PATH, LC_PATH, BLDG_PATH, WIND_PATH, POPULATION_PATH, HIDDEN_DIR, STGNN_WINDOW

best_exp = df_all.iloc[0]['exp_id']
rf      = joblib.load(os.path.join(CKPT_DIR, best_exp, 'rf_model.pkl'))
builder = joblib.load(os.path.join(CKPT_DIR, best_exp, 'feature_builder.pkl'))

# 테스트 데이터 로드
test_data = load_split_data('test', STGNN_WINDOW)
sta2grid  = np.load(os.path.join(HIDDEN_DIR, 'station_to_grid_idx.npy'))

lur_te, lur_names = build_lur_features(
    np.load(NDVI_PATH), np.load(IBI_PATH), np.load(LC_PATH), np.load(BLDG_PATH),
    np.load(POPULATION_PATH) if builder.use_population else None,
    target_global_idx=test_data['global_idx'],
)
lur_te_sta = lur_te[:, sta2grid, :]
temp_te    = build_temporal_features(test_data['timestamps'])

h_te_f, lur_te_f, temp_te_f, pm_te_f = flatten_station_data(test_data, lur_te_sta, temp_te)

X_te = builder.assemble(
    h_te_f   if builder.use_hidden   else None,
    lur_te_f if builder.use_lur      else None,
    temp_te_f if builder.use_temporal else None,
)

y_pred    = rf.predict(X_te)
y_true    = pm_te_f
residuals = y_pred - y_true

mae  = float(np.mean(np.abs(residuals)))
rmse = float(np.sqrt(np.mean(residuals**2)))
r2   = float(1 - np.sum(residuals**2) / np.sum((y_true - y_true.mean())**2))
print(f'[{best_exp}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
vmin, vmax = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
h2d = ax1.hist2d(y_true, y_pred, bins=80, cmap='Blues', range=[[vmin,vmax],[vmin,vmax]])
plt.colorbar(h2d[3], ax=ax1, label='Count')
ax1.plot([vmin,vmax],[vmin,vmax], 'r--', linewidth=1.5, label='Perfect')
ax1.set_xlabel('실측 PM10'); ax1.set_ylabel('예측 PM10')
ax1.set_title(f'예측 vs 실측  MAE={mae:.4f}  R²={r2:.4f}  [{best_exp}]')
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(residuals, bins=60, color='#3498db', edgecolor='white', density=True, alpha=0.85)
xr = np.linspace(residuals.min(), residuals.max(), 300)
ax2.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()), 'r-')
ax2.axvline(0, color='black'); ax2.set_title('잔차 분포')
ax2.set_xlabel('잔차'); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
res_s = np.sort(residuals); n = len(res_s)
tq = stats.norm.ppf((np.arange(1,n+1)-0.5)/n)
ax3.scatter(tq, res_s, s=1, alpha=0.2, color='#3498db')
q1,q3 = np.percentile(residuals,[25,75]); tq1,tq3 = stats.norm.ppf([.25,.75])
sl = (q3-q1)/(tq3-tq1)
ax3.plot([tq.min(),tq.max()],[sl*tq.min()+q1-sl*tq1, sl*tq.max()+q1-sl*tq1],'r-')
ax3.set_title('Q-Q Plot'); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 2])
ax4.scatter(y_pred, residuals, s=2, alpha=0.1, color='#2c3e50')
ax4.axhline(0, color='red', linewidth=1.5)
ax4.axhline(residuals.mean()+2*residuals.std(), color='orange', linestyle='--')
ax4.axhline(residuals.mean()-2*residuals.std(), color='orange', linestyle='--', label='±2σ')
ax4.set_title('잔차 vs 예측값'); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

plt.suptitle(f'잔차 분석 — {best_exp}', fontsize=13)
plt.show()

## 6. Grid PM10 예측 지도

In [ ]:
# grid_pm10_test.npy 가 있으면 시각화
grid_npy = os.path.join(CKPT_DIR, best_exp, 'grid_pm10_test.npy')

if not os.path.exists(grid_npy):
    print(f'grid 추론 파일 없음. 생성하려면:')
    print(f'  python3 inference_grid.py --exp {best_exp}')
else:
    from config import GRID_CSV_PATH
    grid_csv = pd.read_csv(GRID_CSV_PATH)
    pm_grid  = np.load(grid_npy)   # (T, G)
    mean_pm  = pm_grid.mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    ax = axes[0]
    sc = ax.scatter(grid_csv['lon'], grid_csv['lat'], c=mean_pm,
                    cmap='YlOrRd', s=2, linewidths=0, alpha=0.9,
                    vmin=mean_pm.min(), vmax=np.percentile(mean_pm, 99))
    plt.colorbar(sc, ax=ax, label='PM10 (μg/m³)')
    ax.set_title(f'서울 PM10 시간 평균 분포\n[{best_exp}]', fontsize=11)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    ax = axes[1]
    ax.hist(mean_pm, bins=50, color='#e74c3c', edgecolor='white', alpha=0.85)
    ax.set_xlabel('PM10 (μg/m³)'); ax.set_ylabel('격자 수')
    ax.set_title('Grid PM10 분포 히스토그램')
    ax.grid(alpha=0.3)

    plt.suptitle(f'Grid-level PM10 예측 — {best_exp}', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'PM10 통계: mean={mean_pm.mean():.2f}  std={mean_pm.std():.2f}  '
          f'min={mean_pm.min():.2f}  max={mean_pm.max():.2f}')